In [1]:
import sys
import os

from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

sys.path.append("../../")
from src.lib.mlflow import MlflowHandler
from src.main import _ensure_java_home
mlflow_handler = MlflowHandler()

In [2]:
spark_app_name = os.getenv("SPARK_APP_NAME")
spark_master_url = os.getenv("SPARK_MASTER_URL")
postgres_url = os.getenv("POSTGRES_URL")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
_ensure_java_home()

required = {
        "POSTGRES_URL": postgres_url,
        "POSTGRES_USER": postgres_user,
        "POSTGRES_PASSWORD": postgres_password,
    }

spark = (
        SparkSession.builder.appName(spark_app_name)
        .master(spark_master_url)
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.8")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

25/10/30 17:41:55 WARN Utils: Your hostname, MacBook-Air-de-Yose.local resolves to a loopback address: 127.0.0.1; using 10.48.74.89 instead (on interface en0)
25/10/30 17:41:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/yosesotomayor/.ivy2/cache
The jars for the packages stored in: /Users/yosesotomayor/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c3764df-0cb6-4994-b88f-1233d82c90f9;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.8 in central
	found org.checkerframework#checker-qual;3.49.5 in central
:: resolution report :: resolve 167ms :: artifacts dl 7ms
	:: modules in use:
	org.checkerframework#checker-qual;3.49.5 from central in [default]
	org.postgresql#postgresql;42.7.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------

In [3]:
# --- 1. Configuración del Lote ---
BATCH_SIZE = 100000 
MIN_ID = 1
MAX_ID = 31788324

OUTPUT_PATH = "file://Users/yosesotomayor/Desktop/datos_parquet"
# O en la nube: "s3a://mi-bucket/datos_parquet"

jdbc_options = {
    "url": str(postgres_url),
    "user": str(postgres_user),
    "password": str(postgres_password),
    "driver": "org.postgresql.Driver",
    "fetchsize": "10000"
}

print(f"Iniciando volcado a Parquet en: {OUTPUT_PATH}")
print(f"Tamaño de lote: {BATCH_SIZE} filas")

current_id = MIN_ID
total_rows = 0

while current_id <= MAX_ID:
    
    query = f"""
    (
        SELECT * FROM transactions
        WHERE id >= {current_id} AND id < {current_id + BATCH_SIZE}
    ) AS lote_actual
    """
    
    try:
        df_lote = spark.read \
            .format("jdbc") \
            .options(**jdbc_options) \
            .option("dbtable", query) \
            .load()

        count_lote = df_lote.count()
        
        if count_lote > 0:
            df_lote.write.mode("append").parquet(OUTPUT_PATH)
            
            total_rows += count_lote
            print(f"Lote {current_id} a {current_id + BATCH_SIZE - 1} procesado. {count_lote} filas. Total: {total_rows}")
        else:
            print(f"Lote {current_id} a {current_id + BATCH_SIZE - 1} sin datos. Saltando...")

    except Exception as e:
        print(f"Error procesando el lote {current_id}: {e}")
        
    finally:
        try:
            df_lote.unpersist()
        except:
            print("Ups")
        spark.catalog.clearCache() 

    current_id += BATCH_SIZE

print(f"¡Volcado completado! Total de filas: {total_rows}")

Iniciando volcado a Parquet en: file://Users/yosesotomayor/Desktop/datos_parquet
Tamaño de lote: 100000 filas
Error procesando el lote 1: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 100001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 200001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 300001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 400001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 500001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 600001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///
Error procesando el lote 700001: Wrong FS: file://Users/yosesotomayor/Desktop/datos_parquet, expected: file:///

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yosesotomayor/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 